# Multi-Head ConvNeXt V2 Training on Kaggle
Make sure to attach your datasets to this notebook by clicking **Add Input** (top right):
1. `Classified_OCT_Data`


In [2]:
!rm -rf OCT-Analyser-Capstone && git clone -b dev --depth 1 https://github.com/Nikhil-Mundhra/OCT-Analyser-Capstone.git
%cd OCT-Analyser-Capstone

Cloning into 'OCT-Analyser-Capstone'...
remote: Enumerating objects: 4509, done.
remote: Counting objects: 100% (4509/4509), done.
remote: Compressing objects: 100% (1021/1021), done.
remote: Total 4509 (delta 3468), reused 4369 (delta 3454), pack-reused 0 (from 0)
Receiving objects: 100% (4509/4509), 19.57 MiB | 28.63 MiB/s, done.
Resolving deltas: 100% (3468/3468), done.
/kaggle/working/OCT-Analyser-Capstone


In [3]:
# Install requirements silently via uv for faster installs
!pip install -q uv
!UV_NO_PROGRESS=1 uv pip install --system -q -r image-classification-model-training/requirements.txt
!UV_NO_PROGRESS=1 uv pip install --system -q pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 63.9 MB/s eta 0:00:00:00:0100:01


In [4]:
import os
# Quick sanity check to find the exact folder names Kaggle gave your datasets
!ls /kaggle/input/datasets/nikhilmundhra/classified-oct-v2/Classified

 Dataset_Analysis_Report.md	 'Normal (Healthy)'
'Diabetic Complications'	 'Vascular Occlusions (Blockages)'
'Fluid Accumulation'		 'Vitreomacular and Structural Disorders'
'Macular Degeneration Spectrum'


In [11]:
import os
# Stage dataset into local /tmp NVMe disk for maximum reading speed
!mkdir -p /tmp/Classified
!cp -rn /kaggle/input/datasets/nikhilmundhra/classified-oct-v2/Classified/* /tmp/Classified/

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["OCT_DATA_ROOT"] = "/tmp/Classified"
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"

In [ ]:
!torchrun --nproc_per_node=2 image-classification-model-training/scripts/train_convnext.py \n    --config "image-classification-model-training/config/hierarchy.yaml" \n    --batch-size 16 \n    --accum-steps 1 \n    --num-workers 2 \n    --save-steps 2250 \n    --epochs-warmup 3 \n    --epochs-finetune 20 \n    --hf-repo "NMundhra/OCT-Classification-Model"

### Zip Outputs for Easy Download
Run this cell to package both the checkpoints and logs into a single zip file for downloading.

In [ ]:
# Zip the correct checkpoints directory!
!zip -r /kaggle/working/training_outputs.zip checkpoints/ logs/
print("\n✅ Outputs zipped successfully! You can download `training_outputs.zip` from the Kaggle Output pane on the right.")